In [160]:
from dataclasses import dataclass
from enum import IntEnum
from typing import Optional
from copy import deepcopy
import random

# Utility Bitboard Functions

In [161]:
def get_bit(bitboard: int, square: int) -> int:
    return bitboard & (1 << square)

def set_bit(bitboard: int, square: int) -> int:
    return bitboard | (1 << square)

def pop_bit(bitboard: int, square: int) -> int:
    return bitboard & ~(1 << square)


In [162]:
class Square(IntEnum):
    a4 = 0
    b4 = 1
    c4 = 2
    d4 = 3
    a3 = 4
    b3 = 5
    c3 = 6
    d3 = 7
    a2 = 8
    b2 = 9
    c2 = 10
    d2 = 11
    a1 = 12
    b1 = 13
    c1 = 14
    d1 = 15

    def __str__(self):
        rank = 4 - (self // 4)
        file = self % 4
        return "abcd"[file] + str(rank)

def print_bitboard(bitboard: int) -> None:
    print()
    for rank in range(4):
        for file in range(4):
            square = rank * 4 + file
            if file == 0:
                print(f"  {4-rank} ", end="")
            print(f" {int(bool(get_bit(bitboard, square)))}", end = "")
        print()
    print("\n     a b c d")

print_bitboard(1 << 3) 
print_bitboard((1 << 4) + (1 << 3)) 
print_bitboard(set_bit(0, 0 * 4 + 1))



  4  0 0 0 1
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 1
  3  1 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d


In [163]:
for i in range(16):
    print_bitboard(1 << i)


  4  1 0 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 1 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 1
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  1 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 1 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 1 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 1
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  1 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  0 0 1 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  0 0 0 1
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  1 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 1 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 1 0

     a b

## Least Significant Bit Finder

In [164]:
# how the lsb finder works

bitboard = (1 << 2) + (1 << 12) + (1 << 10)
print_bitboard(bitboard)
print_bitboard(-bitboard) 
# ok so it seems because of 2's complement magic, the intersection x & -x agrees only at the lsb
print_bitboard((bitboard & -bitboard)-1)
# then by subtracting 1 it turns all the preceding 0's to 1's and we can bitcount that


  4  0 0 1 0
  3  0 0 0 0
  2  0 0 1 0
  1  1 0 0 0

     a b c d

  4  0 0 1 1
  3  1 1 1 1
  2  1 1 0 1
  1  0 1 1 1

     a b c d

  4  1 1 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d


In [165]:
def get_ls1b_index(bitboard: int) -> Optional[int]:
    if bitboard:
        return ((bitboard & -bitboard) - 1).bit_count()
    else:
        return None

get_ls1b_index((1 << 4) + (1 << 2))

2

# Game State

In [166]:
class Piece(IntEnum):
    K = 0
    W = 1
    M = 2
    F = 3
    P = 4
    k = 5
    w = 6
    m = 7
    f = 8
    p = 9

class Side(IntEnum):
    White = 0
    Black = 1
    Both = 2

ASCII_PIECES = ["K", "W", "M", "F", "P", "k", "w", "m", "f", "p"]

@dataclass
class GameState:
    bitboards: list[int] # White King, Wazir, Ma, Ferz, Pawn + Black King, Wazir, Ma, Ferz, Pawn 
    occupancies: list[int] # white black 
    former_pawns: int
    side: int
    inventory: list[dict[Piece, int]]

    def print_board(self):
        print()
        for rank in range(4):
            for file in range(4):
                square = rank * 4 + file
                if file == 0:
                    print(f"  {4-rank} ", end="")

                piece = -1
                for bb_piece in range(10):
                    if get_bit(self.bitboards[bb_piece], square):
                        piece = bb_piece 
                
                print(f" {ASCII_PIECES[piece] if piece != -1 else '.'}", end = "")
            print()
        print("\n     a b c d")
        print()
        print(f"{self.side} to play")

        for piece, count in self.inventory[0].items():
            if count > 0: print(f"  {ASCII_PIECES[piece]}: {count}")
        for piece, count in self.inventory[1].items():
            if count > 0: print(f"  {ASCII_PIECES[piece]}: {count}")
            
def default_game_state() -> GameState:
    return GameState(bitboards=[
            1 << Square.a1,
            1 << Square.b1,
            1 << Square.c1,
            1 << Square.d1,
            1 << Square.a2,
            1 << Square.d4,
            1 << Square.c4,
            1 << Square.b4,
            1 << Square.a4,
            1 << Square.d3
        ],
        occupancies=[
            (1 << 12) | (1 << 13) | (1 << 14) | (1 << 15) | (1 << 8),
            (1 << 3) | (1 << 2) | (1 << 1) | (1 << 0) | (1 << 7),
            # (1 << 12) | (1 << 13) | (1 << 14) | (1 << 15) | (1 << 8) | (1 << 3) | (1 << 2) | (1 << 1) | (1 << 0) | (1 << 7)
        ],
        former_pawns = 0,
        side = 0, # white to play,
        inventory = [
            {Piece.W: 0, Piece.F: 0, Piece.M: 0, Piece.P: 0}, 
            {Piece.w: 0, Piece.f: 0, Piece.m: 0, Piece.p: 0}, 
        ] # White Inventory, Black Inventory
    )
def create_game_state(piece_map: dict[Piece, Square], side: int, former_pawns: int = 0, inventory = None) -> GameState:
    bitboards = [0 for _ in range(10)]
    occupancies = [0, 0]
    for piece, square in piece_map.items():
        bitboards[piece] = set_bit(bitboards[piece], square)
        occupancies[piece // 5] = set_bit(occupancies[piece // 5], square)

    if inventory is None:
        inventory = [
            {Piece.W: 0, Piece.F: 0, Piece.M: 0, Piece.P: 0}, # test placement
            {Piece.w: 0, Piece.f: 0, Piece.m: 0, Piece.p: 0}, 
        ] #     White Inventory, Black Inventory
    return GameState(
        bitboards=bitboards, occupancies = occupancies, side = side, former_pawns=former_pawns, inventory = inventory
    )


default_game_state().print_board()


  4  f m w k
  3  . . . p
  2  P . . .
  1  K W M F

     a b c d

0 to play


# Generating Attack Masks

For most pieces, the squares that a piece can attack are just the squares a piece can move. The pawn however moves differently to how it attacks. 

Our attack masks are functions that map from one of 16 squares to a `u16` bitboard 
- Since there are only 16 possible inputs, we can store the attack masks as a precomputed table.



In [167]:
NOT_A_FILE = 0b1110_1110_1110_1110
NOT_D_FILE = 0b0111_0111_0111_0111
U16_MAX = 1 << 16 # Unlike most systems programming languages, Python int's automatically resize

## Wazir Attacks

In [168]:
# Recall that the squares are numbered. 

#  0  1  2  3
#  4  5  6  7
#  8  9 10 11
# 12 13 14 15

# Therefore
## >> 4 moves 1 square down e.g. 5->1
## << 4 moves 1 square up e.g. 5->9
## >> 1 moves 1 square left e.g. 5->4
## << 1 moves 1 square right e.g. 5->6


In [169]:
def mask_wazir_attacks(square: int) -> int:
    attacks = 0
    bitboard = 0
    bitboard = set_bit(bitboard, square)

    if (bitboard >> 4): attacks |= (bitboard >> 4) # Goes up 
    if ((bitboard >> 1) & NOT_D_FILE): attacks |= (bitboard >> 1)
    if (bitboard << 4): attacks |= (bitboard << 4) % U16_MAX 
    if ((bitboard << 1) & NOT_A_FILE): attacks |= (bitboard << 1) % U16_MAX 
    
    return attacks

square = Square.c4
print_bitboard(1 << square)
print_bitboard(mask_wazir_attacks(square))


  4  0 0 1 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 1
  3  0 0 1 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d


### Note

To understand why we need the `NOT_A_FILE` and `NOT_D_FILE` mask, consider generating a left attack for a piece on the a file versus on the b file.

In [170]:
square = square.b2
bitboard = set_bit(0, square)
print_bitboard(bitboard) # piece position
attacks = (bitboard >> 1)
print_bitboard(attacks) # piece attack

# Works as expected


  4  0 0 0 0
  3  0 0 0 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  1 0 0 0
  1  0 0 0 0

     a b c d


In [171]:
square = square.a2
bitboard = set_bit(0, square)
print_bitboard(bitboard) # piece position
attacks = (bitboard >> 1)
print_bitboard(attacks) # piece attack

# Piece on a2 can now somehow reach d3
# because bitshifting down moves the 1 bit back so before the set bit was 1 << 8 now it is 1 << 7 which appears to be wrapped around

#  0  1  2  3
#  4  5  6  7
#  8  9 10 11
# 12 13 14 15




  4  0 0 0 0
  3  0 0 0 0
  2  1 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 1
  2  0 0 0 0
  1  0 0 0 0

     a b c d


In [172]:
# correct behaviour is to check that when bitshifting right (to get a left attack) we don't wrap around onto the D_FILE
# and when bitshifting left (to get a right attack) we don't wrap around onto the A_FILE

# this isn't a problem for vertical moves because then the set bit get shifted past 0 or past U16_MAX

square = square.a2
bitboard = set_bit(0, square)
print_bitboard(bitboard) # piece position
attacks = 0
if (bitboard >> 1) & NOT_D_FILE: attacks |= (bitboard >> 1)
print_bitboard(attacks) # piece attack


  4  0 0 0 0
  3  0 0 0 0
  2  1 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d


## Ferz Attacks

In [173]:
def mask_ferz_attacks(square: int) -> int:
    attacks = 0
    bitboard = 0
    bitboard = set_bit(bitboard, square)

    if ((bitboard >> 5) & NOT_D_FILE): attacks |= (bitboard >> 5)
    if ((bitboard >> 3) & NOT_A_FILE): attacks |= (bitboard >> 3)
    if ((bitboard << 5) & NOT_A_FILE): attacks |= (bitboard << 5) % U16_MAX 
    if ((bitboard << 3) & NOT_D_FILE): attacks |= (bitboard << 3) % U16_MAX 
    
    return attacks

square = Square.b2
print_bitboard(1 << square) # Piece position 
print_bitboard(mask_ferz_attacks(square)) # Piece attacks


  4  0 0 0 0
  3  0 0 0 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  1 0 1 0
  2  0 0 0 0
  1  1 0 1 0

     a b c d


## King Attacks

In [174]:
def mask_king_attacks(square: int) -> int:
    attacks = 0
    bitboard = 0
    bitboard = set_bit(bitboard, square)

    if (bitboard >> 4): attacks |= (bitboard >> 4)
    if ((bitboard >> 5) & NOT_D_FILE): attacks |= (bitboard >> 5)
    if ((bitboard >> 3) & NOT_A_FILE): attacks |= (bitboard >> 3)
    if ((bitboard >> 1) & NOT_D_FILE): attacks |= (bitboard >> 1)
    if (bitboard << 4): attacks |= (bitboard << 4) % U16_MAX 
    if ((bitboard << 5) & NOT_A_FILE): attacks |= (bitboard << 5) % U16_MAX
    if ((bitboard << 3) & NOT_D_FILE): attacks |= (bitboard << 3) % U16_MAX
    if ((bitboard << 1) & NOT_A_FILE): attacks |= (bitboard << 1) % U16_MAX
    
    return attacks

square = Square.a1
print_bitboard(1 << square)
print_bitboard(mask_king_attacks(square))
print(mask_king_attacks(square).bit_count())


  4  0 0 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  1 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  1 1 0 0
  1  0 1 0 0

     a b c d
3


## Ma Attacks

In [175]:
def generate_ma_attacks_on_the_fly(square: int, block_mask: int):
    attacks = 0
    
    tr = square // 4
    tf = square % 4

    deltas = {
        (0, 1): [(1, 2), (-1, 2)],
        (1, 0): [(2, 1), (2, -1)],
        (0, -1): [(1, -2), (-1, -2)],
        (-1, 0): [(-2, -1), (-2, 1)]
    }
    for (block_dr, block_df), attack_list in deltas.items():
        block_r = tr + block_dr
        block_f = tf + block_df
        block_square = 4 * block_r + block_f 

        if 0 <= block_square and block_square < 16 and (1 << (block_square) & block_mask):
            continue
        for dr, df in attack_list:
            r = tr + dr
            f = tf + df
            if 0 <= r and r < 4 and 0 <= f and f < 4:
                attacks |= 1 << (4 * r + f)
    return attacks

square = Square.a2
blockers = (1 << Square.b2) | (1 << Square.d2)
print_bitboard(set_bit(0, square))
print_bitboard(blockers)
print_bitboard(generate_ma_attacks_on_the_fly(
    square, blockers
))

# We have a function that can generate on the fly but because it uses some for loops and what not, it may not be desirable performance wise
# We will look at an optimisation called magic tables later
# Our function takes in a square \in [0, ..., 16] and blockers \in [0, ..., U16_MAX] representing the location of all other pieces
# however given the location of the square, the majority of the information in blockers is irrelevant. 
# We only care about the pattern of blockers 1 square orthogonally adjacent to square
# Given there are 4 such possible blocking locations, there are only 2**4 patterns of blocking locations
# So really the crux of our function can be compressed to map square \in [0, ..., 16] and pattern \in [set of size 16] as input


  4  0 0 0 0
  3  0 0 0 0
  2  1 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  0 1 0 1
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d


## Pawn Attacks

In [176]:
def mask_pawn_attacks(side: int, square: int) -> int:
    attacks = 0
    bitboard = 0
    bitboard = set_bit(bitboard, square)

    if side == Side.White: # White
        if ((bitboard >> 3) & NOT_A_FILE): attacks |= (bitboard >> 3)
        if ((bitboard >> 5) & NOT_D_FILE): attacks |= (bitboard >> 5)
    else:
        if ((bitboard << 3) & NOT_D_FILE): attacks |= (bitboard << 3)
        if ((bitboard << 5) & NOT_A_FILE): attacks |= (bitboard << 5)
    return attacks

square = 8
print_bitboard(1 << square)
print_bitboard(mask_pawn_attacks(1, square))


  4  0 0 0 0
  3  0 0 0 0
  2  1 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 1 0 0

     a b c d


# Attack Tables

In [177]:
# PAWN_ATTACKS = [[0 for _ in range(16)] for _ in range(2)]
KING_ATTACKS = [0 for _ in range(16)]
WAZIR_ATTACKS = [0 for _ in range(16)]
FERZ_ATTACKS = [0 for _ in range(16)]
PAWN_ATTACKS = [[0 for _ in range(16)], [0 for _ in range(16)]]

for square in range(16):
    KING_ATTACKS[square] = mask_king_attacks(square)
    WAZIR_ATTACKS[square] = mask_wazir_attacks(square)
    FERZ_ATTACKS[square] = mask_ferz_attacks(square)
    PAWN_ATTACKS[0][square] = mask_pawn_attacks(0, square) # White
    PAWN_ATTACKS[1][square] = mask_pawn_attacks(1, square) # Black

# Magic Tables

In [178]:
occupancy = 0
occupancy = set_bit(occupancy, Square.b4)
occupancy = set_bit(occupancy, Square.c3)
occupancy = set_bit(occupancy, Square.b1)
occupancy = set_bit(occupancy, Square.c1)
occupancy = set_bit(occupancy, Square.d1)
print_bitboard(occupancy) # occupancy bitboard

square = Square.b3 # location of example Ma
blockers = mask_wazir_attacks(square)
print_bitboard(blockers)

print_bitboard(blockers & occupancy)


  4  0 1 0 0
  3  0 0 1 0
  2  0 0 0 0
  1  0 1 1 1

     a b c d

  4  0 1 0 0
  3  1 0 1 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  0 0 1 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d


In [179]:
def generate_magic():
    return random.randint(0, 2**15-1) & random.randint(0, 2**15-1) & random.randint(0, 2**15-1) & random.randint(0, 2**15-1)

def find_magic_number(square: Square) -> Optional[int]:
    blocking_patterns = []
    subset = 0
    blockers = mask_wazir_attacks(square)
    # print_bitboard(blockers)
    while True:
        subset = (subset - blockers) & blockers
        blocking_patterns.append(subset)

        if subset == 0:
            break

    relevant_bits = 2

    attacks = []
    for pattern in blocking_patterns:
        attacks.append(generate_ma_attacks_on_the_fly(square, pattern))

    while True:
        # print("NEW")
        magic = generate_magic()
        used_attacks = [0 for _ in range(4)] # there are at most 4 attack squares for a Ma. Only 2 blockers are relevant. There are hence 2**2 unique attacks patterns per square
        fail = False
        for idx, pattern in enumerate(blocking_patterns):
            magic_index = (((pattern * magic) % U16_MAX) >> (16 - relevant_bits))  
            if used_attacks[magic_index] == 0:
                used_attacks[magic_index] = attacks[idx]
            elif used_attacks[magic_index] == attacks[idx]: # constructive collision: blocker configs that map to the same attack pattern (i.e. non-injectively)
                pass
            else:
                fail = True
                break
        if not fail:
            return magic



In [180]:
square = Square.b3
print_bitboard(1 << square) # Ma Location


  4  0 0 0 0
  3  0 1 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d


In [181]:
blocking_patterns = []
subset = 0
blockers = mask_wazir_attacks(square)
# print_bitboard(blockers)
while True:
    subset = (subset - blockers) & blockers
    blocking_patterns.append(subset)

    if subset == 0:
        break

print(len(blocking_patterns))
for pattern in blocking_patterns:
    print_bitboard(pattern)

16

  4  0 1 0 0
  3  0 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  1 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  1 0 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 1 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  0 0 1 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  1 0 1 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  1 0 1 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 0 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  0 0 0 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  1 0 0 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  1 0 0 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 1 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  0 0 1 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  1 0 1 0
  2  0 1 0 0
  1  0 0 0 0

     a b c d

  4  0 1 0 0
  3  1 0 1 0
  2  0 1 0 0
  1  0 0 0 0

     

In [182]:
MA_ATTACKS = [[0 for _ in range(4)] for _ in range(16)]
MAGICS = [] 
for rank in range(4):
    for file in range(4):
        square = 4 * rank + file
        magic = find_magic_number(square)
        print(Square(square), magic)
        MAGICS.append(magic)
        blockers = mask_wazir_attacks(square) 
        relevant_bits = 2
        subset = 0
        while True:
            subset = (subset - blockers) & blockers
            magic_index = (((subset * magic) % U16_MAX) >> (16 - relevant_bits))
            MA_ATTACKS[square][magic_index] = generate_ma_attacks_on_the_fly(square, subset)

            if subset == 0:
                break



a4 17416
b4 5120
c4 8769
d4 8320
a3 640
b3 320
c3 1044
d3 784
a2 2080
b2 1040
c2 320
d2 304
a1 8260
b1 34
c1 20
d1 10


In [183]:
def generate_ma_attacks(square: Square, block_mask: int) -> int:
    block_mask &= WAZIR_ATTACKS[square]
    block_mask = (block_mask * MAGICS[square]) % U16_MAX
    magic_index = block_mask >> (16 - 2) # high two bits

    attacks = MA_ATTACKS[square]
    return attacks[magic_index]

square = Square.a2
occupancy = (1 << 4) | (1 << 2) | (1 << 5)
print_bitboard(occupancy) 
print_bitboard(generate_ma_attacks(square, occupancy))


  4  0 0 1 0
  3  1 1 0 0
  2  0 0 0 0
  1  0 0 0 0

     a b c d

  4  0 0 0 0
  3  0 0 1 0
  2  0 0 0 0
  1  0 0 1 0

     a b c d


# Pseudolegal Move Gen

In [184]:
def enemy_attack_mask(game_state: GameState, side: int):
    # if side == 0, then we look at what squares are attacked by side == 1 i.e. Black
    attack_mask = 0
    occupancies = game_state.occupancies

    # KING
    bitboard = game_state.bitboards[Piece.k if side == 0 else Piece.K]
    while bitboard:
        source_square = get_ls1b_index(bitboard)

        attacks = KING_ATTACKS[source_square]
        attack_mask |= attacks
        bitboard = pop_bit(bitboard, source_square)
    
    # WAZIR 
    bitboard = game_state.bitboards[Piece.w if side == 0 else Piece.W]
    while bitboard:
        source_square = get_ls1b_index(bitboard)

        attacks = WAZIR_ATTACKS[source_square]
        attack_mask |= attacks
        bitboard = pop_bit(bitboard, source_square)
    
    # FERZ 
    bitboard = game_state.bitboards[Piece.f if side == 0 else Piece.F]
    while bitboard:
        source_square = get_ls1b_index(bitboard)

        attacks = FERZ_ATTACKS[source_square]
        attack_mask |= attacks
        bitboard = pop_bit(bitboard, source_square)
    
    # MA 
    both = occupancies[0] | occupancies[1] & ~game_state.bitboards[Piece.K if side == 0 else Piece.k] # Since it is possible for the King to be blocking the Ma, we need to check if the King moves away from blocking the Ma and into check
    bitboard = game_state.bitboards[Piece.m if side == 0 else Piece.M]
    while bitboard:
        source_square = get_ls1b_index(bitboard)

        attacks = generate_ma_attacks(source_square, both) 
        attack_mask |= attacks
        bitboard = pop_bit(bitboard, source_square)

    # PAWN
    bitboard = game_state.bitboards[Piece.p if side == 0 else Piece.P]
    while bitboard:
        source_square = get_ls1b_index(bitboard)

        attacks = PAWN_ATTACKS[1 - side][source_square]
        attack_mask |= attacks
        bitboard = pop_bit(bitboard, source_square)

    return attack_mask

game_state = default_game_state()
game_state.print_board()
print_bitboard(enemy_attack_mask(game_state, game_state.side))

game_state = GameState(bitboards=[
        1 << Square.a1,
        0,
        1 << Square.b2,
        0,
        1 << Square.c3,
        1 << Square.b3,
        1 << Square.c2,
        0,
        0,
        0
    ],
    occupancies=[
        (1 << Square.a1) | (1 << Square.b2) | (1 << Square.c3),
        (1 << Square.d3) | (1 << Square.c2)
    ],
    side = 1, # white to play,
    former_pawns = 0,
    inventory = [
        {Piece.W: 0, Piece.F: 0, Piece.M: 0, Piece.P: 0}, # test placement
        {Piece.w: 0, Piece.f: 0, Piece.m: 0, Piece.p: 0}, 
    ] # White Inventory, Black Inventory
)
game_state.print_board()
print_bitboard(enemy_attack_mask(game_state, game_state.side))


  4  f m w k
  3  . . . p
  2  P . . .
  1  K W M F

     a b c d

0 to play

  4  0 1 1 1
  3  0 1 1 1
  2  1 0 1 0
  1  0 0 0 0

     a b c d

  4  . . . .
  3  . k P .
  2  . M w .
  1  K . . .

     a b c d

1 to play

  4  1 1 1 1
  3  0 0 0 0
  2  1 1 0 0
  1  0 1 0 0

     a b c d


In [185]:
@dataclass 
class Move:
    source: Optional[Square]
    target: Square
    piece: Piece
    promoted_piece: Optional[Piece]
    capture: bool
    placement: bool # this is redundant because if Source is None, then it must be a Placement
    # bit packing is probs better but I ceebs for now
    
    def __str__(self):
        if self.placement:
            return f"@{ASCII_PIECES[self.piece]}{Square(self.target)}"
        else:
            return f"{ASCII_PIECES[self.piece]}{'x' if self.capture else ''}{Square(self.target)}{f'={ASCII_PIECES[self.promoted_piece]}' if self.promoted_piece else ''}"

def generate_moves(game_state: GameState):
    occupancies = game_state.occupancies
    side = game_state.side
    both = occupancies[0] | occupancies[1] 
    inventory = game_state.inventory[side]

    for piece, count in inventory.items():
        if count == 0: continue
        available_squares = ~both % U16_MAX
        while available_squares:
            available_square = get_ls1b_index(available_squares)
            if piece == Piece.P and available_square <= Square.d4: 
                available_squares = pop_bit(available_squares, available_square)
                continue
            if piece == Piece.p and available_square >= Square.a1: 
                available_squares = pop_bit(available_squares, available_square)
                continue

            # print(int_to_square(available_square), ASCII_PIECES[piece], "Place")
            yield Move(None, available_square, piece, None, False, True)
            available_squares = pop_bit(available_squares, available_square)
            
    both = occupancies[0] | occupancies[1]
    for piece in range(10):
        bitboard = game_state.bitboards[piece]

        # generate King Moves
        if ((piece == Piece.K) if (side == 0) else (piece == Piece.k)):
            while bitboard:
                source_square = get_ls1b_index(bitboard)

                attacks = KING_ATTACKS[source_square] & ~occupancies[side] & ~enemy_attack_mask(game_state, game_state.side)
                while attacks:
                    target_square = get_ls1b_index(attacks)

                    # Quiet Move
                    if not get_bit(occupancies[1 - side], target_square):
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "move")
                        yield Move(source_square, target_square, piece, None, False, False)
                    # Capture 
                    else:
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "capture")
                        yield Move(source_square, target_square, piece, None, True, False)
                    attacks = pop_bit(attacks, target_square)
                bitboard = pop_bit(bitboard, source_square)

        # generate WAZIR Moves
        if ((piece == Piece.W) if (side == 0) else (piece == Piece.w)):
            while bitboard:
                source_square = get_ls1b_index(bitboard)

                attacks = WAZIR_ATTACKS[source_square] & ~occupancies[side] 
                while attacks:
                    target_square = get_ls1b_index(attacks)

                    # Quiet Move
                    if not get_bit(occupancies[1 - side], target_square):
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "move")
                        yield Move(source_square, target_square, piece, None, False, False)
                    # Capture 
                    else:
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "capture")
                        yield Move(source_square, target_square, piece, None, True, False)
                    attacks = pop_bit(attacks, target_square)
                bitboard = pop_bit(bitboard, source_square)
        
        # generate FERZ Moves
        if ((piece == Piece.F) if (side == 0) else (piece == Piece.f)):
            while bitboard:
                source_square = get_ls1b_index(bitboard)
                
                attacks = FERZ_ATTACKS[source_square] & ~occupancies[side] 
                while attacks:
                    target_square = get_ls1b_index(attacks)

                    # Quiet Move
                    if not get_bit(occupancies[1 - side], target_square):
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "move")
                        yield Move(source_square, target_square, piece, None, False, False)
                    # Capture 
                    else:
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "capture")
                        yield Move(source_square, target_square, piece, None, True, False)
                    attacks = pop_bit(attacks, target_square)
                bitboard = pop_bit(bitboard, source_square)

        # generate MA moves
        if ((piece == Piece.M) if (side == 0) else (piece == Piece.m)): 
            while bitboard:
                source_square = get_ls1b_index(bitboard)

                attacks = generate_ma_attacks(source_square, both) & ~occupancies[side]
                while attacks:
                    target_square = get_ls1b_index(attacks)

                    # Quiet Move
                    if not get_bit(occupancies[1 - side], target_square):
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "move")
                        yield Move(source_square, target_square, piece, None, False, False)
                    # Capture 
                    else:
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "capture")                    # 
                        yield Move(source_square, target_square, piece, None, True, False)

                    attacks = pop_bit(attacks, target_square) 
                bitboard = pop_bit(bitboard, source_square)

        # generate PAWN Moves
        if side == 0 and piece == Piece.P:
            while bitboard:
                source_square = get_ls1b_index(bitboard)

                target_square = source_square - 4

                # Quiet Move
                if (not (target_square < Square.a4) and not get_bit(both, target_square)):
                    if (source_square >= Square.a3 and source_square <= Square.d3):
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "Promotion")
                        yield Move(source_square, target_square, piece, Piece.W, False, False)
                        yield Move(source_square, target_square, piece, Piece.F, False, False)
                        yield Move(source_square, target_square, piece, Piece.M, False, False)
                    else:
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "Move")
                        yield Move(source_square, target_square, piece, None, False, False)
                
                attacks = PAWN_ATTACKS[side][source_square] & occupancies[1]
                while attacks:
                    target_square = get_ls1b_index(attacks)
                    if (source_square >= Square.a3 and source_square <= Square.d3):
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "Capture + Promotion")
                        yield Move(source_square, target_square, piece, Piece.W, True, False)
                        yield Move(source_square, target_square, piece, Piece.F, True, False)
                        yield Move(source_square, target_square, piece, Piece.M, True, False)
                    else:
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "Capture")
                        yield Move(source_square, target_square, piece, None, True, False)

                    attacks = pop_bit(attacks, target_square)
                bitboard = pop_bit(bitboard, source_square)
        if side == 1 and piece == Piece.p:
            while bitboard:
                source_square = get_ls1b_index(bitboard)

                both = occupancies[0] | occupancies[1]
                target_square = source_square + 4

                # Quiet Move
                if (not (target_square > Square.d1) and not get_bit(both, target_square)):
                    if (source_square >= Square.a2 and source_square <= Square.d2):
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "Promotion")
                        yield Move(source_square, target_square, piece, Piece.w, False, False)
                        yield Move(source_square, target_square, piece, Piece.f, False, False)
                        yield Move(source_square, target_square, piece, Piece.m, False, False)
                    else:
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "Move")
                        yield Move(source_square, target_square, piece, None, False, False)
                
                attacks = PAWN_ATTACKS[side][source_square] & occupancies[0]
                while attacks:
                    target_square = get_ls1b_index(attacks)
                    if (source_square >= Square.a2 and source_square <= Square.d2):
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "Capture + Promotion")
                        yield Move(source_square, target_square, piece, Piece.w, True, False)
                        yield Move(source_square, target_square, piece, Piece.f, True, False)
                        yield Move(source_square, target_square, piece, Piece.m, True, False)
                    else:
                        # print(int_to_square(source_square), int_to_square(target_square), ASCII_PIECES[piece], "Capture")
                        yield Move(source_square, target_square, piece, None, True, False)

                    attacks = pop_bit(attacks, target_square)
                bitboard = pop_bit(bitboard, source_square)     

# list(generate_moves(default_game_state()))

game_state = GameState(bitboards=[
        1 << Square.a1,
        1 << Square.b1,
        1 << Square.c1,
        1 << Square.d1,
        1 << Square.a2,
        1 << Square.d4,
        1 << Square.c4,
        1 << Square.b4,
        1 << Square.a4,
        1 << Square.d3
    ],
    occupancies=[
        (1 << 12) | (1 << 13) | (1 << 14) | (1 << 15) | (1 << 8),
        (1 << 3) | (1 << 2) | (1 << 1) | (1 << 0) | (1 << 7),
        # (1 << 12) | (1 << 13) | (1 << 14) | (1 << 15) | (1 << 8) | (1 << 3) | (1 << 2) | (1 << 1) | (1 << 0) | (1 << 7)
    ],
    former_pawns=0,
    side = 0, # white to play,
    inventory = [
        {Piece.W: 0, Piece.F: 0, Piece.M: 1, Piece.P: 0}, # test placement
        {Piece.w: 0, Piece.f: 0, Piece.m: 0, Piece.p: 0}, 
    ] # White Inventory, Black Inventory
)
game_state.print_board()
list(generate_moves(game_state))


  4  f m w k
  3  . . . p
  2  P . . .
  1  K W M F

     a b c d

0 to play
  M: 1


[Move(source=None, target=4, piece=<Piece.M: 2>, promoted_piece=None, capture=False, placement=True),
 Move(source=None, target=5, piece=<Piece.M: 2>, promoted_piece=None, capture=False, placement=True),
 Move(source=None, target=6, piece=<Piece.M: 2>, promoted_piece=None, capture=False, placement=True),
 Move(source=None, target=9, piece=<Piece.M: 2>, promoted_piece=None, capture=False, placement=True),
 Move(source=None, target=10, piece=<Piece.M: 2>, promoted_piece=None, capture=False, placement=True),
 Move(source=None, target=11, piece=<Piece.M: 2>, promoted_piece=None, capture=False, placement=True),
 Move(source=12, target=9, piece=0, promoted_piece=None, capture=False, placement=False),
 Move(source=13, target=9, piece=1, promoted_piece=None, capture=False, placement=False),
 Move(source=14, target=5, piece=2, promoted_piece=None, capture=False, placement=False),
 Move(source=14, target=7, piece=2, promoted_piece=None, capture=True, placement=False),
 Move(source=15, target=10,

In [186]:
def make_move(move: Move, game_state: GameState) -> GameState:
    new_state = deepcopy(game_state)

    if move.placement:
        assert new_state.inventory[new_state.side][move.piece] > 0
        new_state.inventory[new_state.side][move.piece] -= 1
        new_state.bitboards[move.piece] = set_bit(new_state.bitboards[move.piece], move.target) # place piece
        new_state.occupancies[new_state.side] = set_bit(new_state.occupancies[new_state.side], move.target) # set occupancy on target
    else:
        if move.capture:
            for piece in range(10):
                if (new_state.bitboards[piece] & (1 << move.target)):
                    assert not ((piece == Piece.K) or (piece == Piece.k))
                    break
            else:
                assert False, "Unreachable"

            if get_bit(new_state.former_pawns, move.target):
                # captured a former pawn
                new_state.inventory[new_state.side][Piece.P if new_state.side == 0 else Piece.p] += 1
                new_state.former_pawns = pop_bit(new_state.former_pawns, move.target)
            else:
                new_state.inventory[new_state.side][(piece + 5) % 10] += 1

            new_state.bitboards[piece] = pop_bit(new_state.bitboards[piece], move.target) # additionally remove enemy piece
            new_state.occupancies[1-new_state.side] = pop_bit(new_state.occupancies[1-new_state.side], move.target) # additionally remove enemy occupancy from target
        
        # Handle Promotions
        if move.promoted_piece is not None:
            new_state.bitboards[move.piece] = pop_bit(new_state.bitboards[move.piece], move.source)
            new_state.bitboards[move.promoted_piece] = set_bit(new_state.bitboards[move.promoted_piece], move.target)
            new_state.former_pawns = set_bit(new_state.former_pawns, move.target)
        else:
            new_state.bitboards[move.piece] = set_bit(pop_bit(new_state.bitboards[move.piece], move.source), move.target) # move piece

        new_state.occupancies[new_state.side] = pop_bit(new_state.occupancies[new_state.side], move.source) # remove occupancy from source
        new_state.occupancies[new_state.side] = set_bit(new_state.occupancies[new_state.side], move.target) # set occupancy on target 

        # Move former pawn 
        if get_bit(new_state.former_pawns, move.source):
            new_state.former_pawns = set_bit(pop_bit(new_state.former_pawns, move.source), move.target) # move piece

    new_state.side = 1 - new_state.side
    return new_state


# Legal Move gen

In [187]:
def detect_check(game_state: GameState, side: int):
    attack_mask = enemy_attack_mask(game_state, side)
    return bool(attack_mask & game_state.bitboards[Piece.K if side == 0 else Piece.k]) 

def generate_legal_moves(game_state):
    for move in generate_moves(game_state):
        if detect_check(make_move(move, game_state), game_state.side):
            continue
        yield move




In [188]:
game_state = create_game_state({
    Piece.K: Square.a1,
    Piece.k: Square.d4,
    Piece.W: Square.b1,
    Piece.m: Square.b3
}, 0, 0, [
    {Piece.W: 0, Piece.F: 0, Piece.M: 0, Piece.P: 1},
    {Piece.w: 0, Piece.f: 0, Piece.m: 0, Piece.p: 0}, 
])
print(game_state.former_pawns)
game_state.print_board()

print("Pseudolegal moves")
for move in generate_moves(game_state):
    print(move)

print("Legal moves")
for move in generate_legal_moves(game_state):
    print(move)

0

  4  . . . k
  3  . m . .
  2  . . . .
  1  K W . .

     a b c d

0 to play
  P: 1
Pseudolegal moves
@Pa3
@Pc3
@Pd3
@Pa2
@Pb2
@Pc2
@Pd2
@Pc1
@Pd1
Ka2
Kb2
Wb2
Wc1
Legal moves
@Pb2
Ka2
Kb2
Wb2


In [189]:
def perft(game_state: GameState, depth: int):
    if depth == 0:
        return 1
    count = 0
    for move in generate_legal_moves(game_state):
        count += perft(make_move(move, game_state), depth - 1)
    return count

for i in range(7):
    print(i, perft(default_game_state(), i))

0 1
1 6
2 33
3 246
4 1939
5 17231
6 153180
